<a href="https://colab.research.google.com/github/AmanuelDaget/YOLOv12-Object-Recognition-from-Remote-Sensing-images/blob/main/YOLOv12_Object_Recognition_from_Remote_Sensing_images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Object Recognition using YOLOv12 from Remote Sensing images**

**Install important libraries**

In [3]:
# !pip install ultralytics torchvision pyyaml opencv-python -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.4 MB/s eta 0:00:00


**Import libraries**

In [7]:
import os, shutil, yaml, random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from ultralytics import YOLO
from google.colab import drive

**MOUNT GOOGLE DRIVE**

In [7]:
drive.mount('/content/drive')

Mounted at /content/drive


**CONFIG**

In [8]:
DRIVE = '/content/drive/MyDrive/Colab Notebooks/Computer Vision and Image Processing/Object-detection-using-YOLOv12'
NUM_CLASSES = 15
CLASSES = [
    "car","truck","van","long_vehicle","bus",
    "airliner","propeller_aircraft","trainer_aircraft",
    "chartered_aircraft","fighter_aircraft",
    "others","stair_truck","pushback_truck",
    "helicopter","boat"
]

**DATASET PREPARATION**

In [6]:
print("DATASET PREPARATION")
def prepare_dataset():
    for split, src_img, src_lbl in [
        ("train", f"{DRIVE}/training/images", f"{DRIVE}/training/labels"),
        ("val",   f"{DRIVE}/test/images",     f"{DRIVE}/test/labels"),
    ]:
        for sub in ["images", "labels"]:
            os.makedirs(f"{BASE}/{sub}/{split}", exist_ok=True)

        imgs = sorted(Path(src_img).glob("*.[Jj][Pp][Gg]")) + \
               sorted(Path(src_img).glob("*.png"))
        for img in imgs:
            shutil.copy(img, f"{BASE}/images/{split}/{img.name}")
            lbl = Path(src_lbl) / img.with_suffix(".txt").name
            if lbl.exists():
                shutil.copy(lbl, f"{BASE}/labels/{split}/{lbl.name}")

    print(f"  Train images : {len(list(Path(f'{BASE}/images/train').iterdir()))}")
    print(f"  Val   images : {len(list(Path(f'{BASE}/images/val').iterdir()))}")

prepare_dataset()

Folder '/content/simd' and all its contents have been deleted.


**YAML CONFIG**

In [ ]:
def read_labels(path):

    if not path.exists():
        return []

    out = []

    for line in path.read_text().splitlines():

        c, x, y, w, h = map(float, line.split())

        out.append((int(c), x, y, w, h))

    return out

def remap_box(cx, cy, bw, bh, tx, ty, W, H):

    x1 = (cx - bw/2) * W
    y1 = (cy - bh/2) * H
    x2 = (cx + bw/2) * W
    y2 = (cy + bh/2) * H

    ix1 = max(x1, tx)
    iy1 = max(y1, ty)

    ix2 = min(x2, tx + TILE)
    iy2 = min(y2, ty + TILE)

    if ix2 <= ix1 or iy2 <= iy1:
        return None

    inter = (ix2 - ix1) * (iy2 - iy1)
    area  = (x2 - x1) * (y2 - y1)

    if inter / (area + 1e-6) < MIN_AREA:
        return None

    return (
        ((ix1 + ix2)/2 - tx) / TILE,
        ((iy1 + iy2)/2 - ty) / TILE,
        (ix2 - ix1) / TILE,
        (iy2 - iy1) / TILE
    )

def tile_dataset(split):

    src_img = ROOT / split / "image"
    src_lbl = ROOT / split / "label"

    dst_img = OUT / split / "images"
    dst_lbl = OUT / split / "labels"

    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    stride = int(TILE * (1 - OVERLAP))

    imgs = list(src_img.glob("*.jpg")) + list(src_img.glob("*.JPG"))

    for img_path in tqdm(imgs, desc=f"{split}"):

        img = cv2.imread(str(img_path))

        if img is None:
            continue

        img = clahe(img)

        H, W = img.shape[:2]

        labels = read_labels(src_lbl / f"{img_path.stem}.txt")

        xs = list(range(0, max(W - TILE, 1), stride)) + [max(W - TILE, 0)]
        ys = list(range(0, max(H - TILE, 1), stride)) + [max(H - TILE, 0)]

        idx = 0

        for ty in ys:
            for tx in xs:

                crop = img[ty:ty+TILE, tx:tx+TILE]

                canvas = np.zeros((TILE, TILE, 3), dtype=np.uint8)

                canvas[:crop.shape[0], :crop.shape[1]] = crop

                new_labels = []

                for c, cx, cy, bw, bh in labels:

                    r = remap_box(cx, cy, bw, bh, tx, ty, W, H)

                    if r:
                        new_labels.append((c, *r))

                name = f"{img_path.stem}_{idx}"

                cv2.imwrite(str(dst_img / f"{name}.jpg"), canvas)

                with open(dst_lbl / f"{name}.txt", "w") as f:

                    for row in new_labels:

                        f.write(
                            f"{row[0]} {row[1]:.6f} {row[2]:.6f} "
                            f"{row[3]:.6f} {row[4]:.6f}\n"
                        )

                idx += 1


**Yaml file**

In [ ]:
def write_yaml():

    text = f"""
path: {OUT}

train: training/images
val: test/images
test: test/images

nc: {len(CLASSES)}

names: {CLASSES}
"""

    YAML.write_text(text)

**TRAIN**

In [ ]:
def train():

    model = YOLO("yolo11s.pt")

    model.train(

        data=str(YAML),

        imgsz=TILE,

        epochs=100,

        batch=16,

        device=0,

        optimizer="AdamW",

        lr0=1e-3,

        weight_decay=5e-4,

        mosaic=0.5,
        mixup=0.1,
        copy_paste=0.3,

        degrees=180,

        flipud=0.5,
        fliplr=0.5,

        hsv_h=0.015,
        hsv_s=0.5,
        hsv_v=0.3,

        multi_scale=True,

        project=str(RUNS),

        name="yolo11_simd",

        exist_ok=True
    )

**Visualize Train Results**

**Test**

**Visualize Test Results**